In [ ]:
# 실행 시간 측정: 단일 AEDTPLT export
import time

start_aedtplt = time.perf_counter()

# 첫 번째 variation만 추출
test_table_1_aedtplt = ParametricTable.iloc[:1]

print("=" * 70)
print(f"⏱️  AEDTPLT Export 시간 측정 (1개 Variation)")
print("=" * 70)

# AEDTPLT export 실행
aedtplt_files_test = export_aedtplt_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_1_aedtplt,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    output_dir=r"D:\KDHe10\e10_example\AEDTPLT_Exports_Test",
    intrinsics={"Time": "0.06s"}
)

elapsed_aedtplt = time.perf_counter() - start_aedtplt

print(f"\n{'='*70}")
print("📊 AEDTPLT 성능 측정 결과")
print(f"{'='*70}")
print(f"✅ 총 소요 시간: {elapsed_aedtplt:.3f} 초")
print(f"📦 생성된 파일: {len(aedtplt_files_test)}개")
print(f"\n💡 전체 {len(ParametricTable)}개 variation 예상 시간:")
print(f"   약 {elapsed_aedtplt * len(ParametricTable):.1f} 초 ({elapsed_aedtplt * len(ParametricTable) / 60:.1f} 분)")
print(f"{'='*70}")

# Maxwell 2D Field Data Export

## Setup: AEDT Connection & Utilities

In [12]:
import ansys.aedt.core
import os

import tempfile
import time
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
from ansys.aedt.core import Desktop

import ansys.aedt.core
from ansys.aedt.core import Desktop
import subprocess
import psutil
import time
import os
import re

def get_aedt_processes_detailed():
    """
    실행 중인 AEDT 프로세스를 상세히 확인합니다.
    
    Returns:
    --------
    list : AEDT 프로세스 정보 리스트
    """
    print("🔍 실행 중인 AEDT 프로세스 검색...")
    
    aedt_processes = []
    try:
        for proc in psutil.process_iter(['pid', 'name', 'cmdline', 'create_time']):
            try:
                pinfo = proc.info
                process_name = pinfo['name'] if pinfo['name'] else ""
                
                # AEDT 관련 프로세스 필터링
                if any(keyword in process_name.lower() for keyword in ['ansysedt', 'aedt']):
                    # 포트 정보 추출 시도
                    ports = []
                    try:
                        connections = proc.connections()
                        for conn in connections:
                            if conn.status == 'LISTEN':
                                ports.append(conn.laddr.port)
                    except (psutil.AccessDenied, psutil.NoSuchProcess):
                        pass
                    
                    aedt_processes.append({
                        'pid': pinfo['pid'],
                        'name': process_name,
                        'cmdline': pinfo['cmdline'] if pinfo['cmdline'] else [],
                        'create_time': time.ctime(pinfo['create_time']),
                        'ports': ports
                    })
                    
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
    
        if aedt_processes:
            print(f"✅ {len(aedt_processes)}개의 AEDT 프로세스 발견:")
            for i, proc in enumerate(aedt_processes):
                print(f"\n📋 프로세스 {i+1}:")
                print(f"   PID: {proc['pid']}")
                print(f"   이름: {proc['name']}")
                print(f"   생성시간: {proc['create_time']}")
                if proc['ports']:
                    print(f"   열린 포트: {proc['ports']}")
                else:
                    print(f"   열린 포트: 없음")
        else:
            print("❌ AEDT 프로세스가 없습니다.")
            
        return aedt_processes
        
    except Exception as e:
        print(f"❌ 프로세스 검색 중 오류: {e}")
        return []

def try_connect_to_existing_desktop():
    """
    기존 AEDT Desktop에 연결을 시도합니다.
    
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    print("🔗 기존 AEDT Desktop 연결 시도...")
    
    try:
        # 방법 1: new_desktop_session=False로 기존 세션에 연결
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=False,
            non_graphical=NG_MODE
        )
        print("✅ 기존 AEDT Desktop에 성공적으로 연결되었습니다!")
        return desktop
        
    except Exception as e:
        print(f"❌ 기존 Desktop 연결 실패: {e}")
        return None

def try_connect_with_ports(port_list):
    """
    특정 포트들을 시도해서 AEDT에 연결합니다.
    
    Parameters:
    -----------
    port_list : list
        시도할 포트 번호 리스트
        
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    AEDT_VERSION='251'
    NG_MODE=False
    for port in port_list:
        try:
            print(f"🔗 포트 {port}로 연결 시도...")
            desktop = Desktop(
                specified_version=AEDT_VERSION,
                new_desktop_session=False,
                port=port,
                non_graphical=NG_MODE
            )
            print(f"✅ 포트 {port}로 성공적으로 연결되었습니다!")
            return desktop
        except Exception as e:
            print(f"❌ 포트 {port} 연결 실패: {e}")
            continue
    
    return None

def get_desktop_connection():
    """
    다양한 방법으로 AEDT Desktop 연결을 시도합니다.
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("=" * 60)
    print("🎯 AEDT Desktop 연결 시도")
    print("=" * 60)
    
    # 1. 기존 Desktop 연결 시도
    desktop = try_connect_to_existing_desktop()
    if desktop:
        return desktop
    
    # 2. 프로세스에서 포트 찾아서 연결 시도
    processes = get_aedt_processes_detailed()
    all_ports = []
    
    for proc in processes:
        all_ports.extend(proc['ports'])
    
    if all_ports:
        desktop = try_connect_with_ports(all_ports)
        if desktop:
            return desktop
    
    # 3. 일반적인 AEDT 포트들 시도
    common_ports = [56800, 56801, 56802, 56803, 56804, 56805]
    print("\n🔍 일반적인 AEDT 포트들 시도...")
    desktop = try_connect_with_ports(common_ports)
    if desktop:
        return desktop
    
    # 4. 새로운 Desktop 세션 생성
    print("\n🆕 새로운 AEDT Desktop 세션을 생성합니다...")
    try:
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
        print("✅ 새로운 AEDT Desktop이 생성되었습니다!")
        return desktop
    except Exception as e:
        print(f"❌ 새 Desktop 생성 실패: {e}")
        return None

def check_current_desktop_status(desktop):
    """
    현재 Desktop의 상태를 확인합니다.
    
    Parameters:
    -----------
    desktop : Desktop
        확인할 Desktop 객체
    """
    if not desktop:
        print("❌ Desktop 객체가 없습니다.")
        return
    
    try:
        print("\n" + "=" * 40)
        print("📊 현재 Desktop 상태:")
        print("=" * 40)
        
        # 기본 정보
        print(f"AEDT 버전: {desktop.aedt_version_id}")
        print(f"프로세스 ID: {desktop.aedt_process_id}")
        
        # 프로젝트 정보
        try:
            projects = desktop.project_list()
            print(f"\n📁 열린 프로젝트 ({len(projects)}개):")
            for i, proj_name in enumerate(projects, 1):
                print(f"  {i}. {proj_name}")
            
            # 활성 프로젝트
            active_proj = desktop.active_project()
            if active_proj:
                proj_name = active_proj.GetName()
                print(f"\n🎯 활성 프로젝트: {proj_name}")
                
                # 디자인 목록
                try:
                    design_list = active_proj.GetTopDesignList()
                    print(f"📐 디자인 ({len(design_list)}개):")
                    for i, design in enumerate(design_list, 1):
                        print(f"  {i}. {design}")
                        
                    # 활성 디자인
                    active_design = desktop.active_design()
                    if active_design:
                        print(f"🎯 활성 디자인: {active_design.GetName()}")
                        print(f"   디자인 타입: {active_design.GetDesignType()}")
                except:
                    print("디자인 정보 가져오기 실패")
            else:
                print("🎯 활성 프로젝트: 없음")
                
        except Exception as e:
            print(f"프로젝트 정보 가져오기 실패: {e}")
            
    except Exception as e:
        print(f"❌ Desktop 상태 확인 중 오류: {e}")

def smart_aedt_connector():
    """
    스마트 AEDT 연결 함수 - 사용자 친화적 인터페이스
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("🚀 스마트 AEDT 연결기를 시작합니다...")
    
    # Desktop 연결 시도
    desktop = get_desktop_connection()
    
    if desktop:
        # 연결 성공 시 상태 확인
        check_current_desktop_status(desktop)
        
        print("\n" + "=" * 60)
        print("🎉 AEDT Desktop 연결이 완료되었습니다!")
        print("💡 다음과 같이 사용할 수 있습니다:")
        print("=" * 60)
        print("# 프로젝트 열기:")
        print("# project = desktop.open_project(r'C:\\path\\to\\your\\project.aedt')")
        print("#")
        print("# Maxwell 객체 생성:")
        print("# m2d = ansys.aedt.core.Maxwell2d(project=desktop, new_desktop=False)")
        print("# m3d = ansys.aedt.core.Maxwell3d(project=desktop, new_desktop=False)")
        print("=" * 60)
        
        return desktop
    else:
        print("❌ AEDT Desktop 연결에 실패했습니다.")
        print("\n🔍 문제 해결 방법:")
        print("1. Ansys AEDT가 설치되어 있는지 확인")
        print("2. AEDT 라이선스가 사용 가능한지 확인")
        print("3. 수동으로 AEDT를 실행한 후 다시 시도")
        return None

# 간단한 사용 함수들
def quick_connect():
    """빠른 연결 - 기존 세션 우선"""
    return try_connect_to_existing_desktop()

def force_new_session():
    """강제로 새 세션 생성"""
    try:
        return Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
    except Exception as e:
        print(f"새 세션 생성 실패: {e}")
        return None


## Load Maxwell 2D Model

In [ ]:
# e10 Model
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [ ]:
aedt_file=r"D:\KDHe10\e10_example\e10_tutorial_ANSYSEM_2D.aedt"
m2d = ansys.aedt.core.Maxwell2d(
    project=aedt_file,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE
)

## Utility: 현재 실행중인 Maxwell2d 객체 자동 연결 함수

In [13]:
def get_running_maxwell2d(aedt_version="2025.2", non_graphical=False):
    """
    현재 실행 중인 AEDT Desktop 세션에 연결하여 활성 Maxwell 2D 디자인의 Maxwell2d 객체를 반환합니다.

    동작 순서:
    1) 기존 Desktop 세션에 연결 (새 세션 생성하지 않음)
    2) 활성 프로젝트/디자인 조회
    3) 활성 디자인이 Maxwell 2D가 아니면, 프로젝트의 디자인 목록에서 Maxwell 2D를 탐색
    4) 찾은 프로젝트/디자인 이름으로 Maxwell2d 객체 attach

    Parameters
    ----------
    aedt_version : str
        AEDT 버전 문자열 (예: "2025.2").
    non_graphical : bool
        비그래픽 모드 여부. 기존 실행 세션에 attach할 때는 보통 False 권장.

    Returns
    -------
    ansys.aedt.core.Maxwell2d or None
        연결된 Maxwell2d 객체. 찾지 못하면 None 반환.
    """
    try:
        from ansys.aedt.core import Desktop, Maxwell2d
    except Exception as e:
        print(f"❌ PyAEDT import 실패: {e}")
        return None

    desktop = None
    try:
        # 기존 세션에 붙기 (새 세션 X)
        desktop = Desktop(
            specified_version=aedt_version,
            new_desktop_session=False,
            non_graphical=non_graphical
        )
    except Exception as e:
        print(f"❌ 기존 Desktop 연결 실패: {e}")
        return None

    # 활성 프로젝트/디자인 가져오기
    try:
        active_proj = desktop.active_project()
        if not active_proj:
            projs = desktop.project_list()
            if not projs:
                print("❌ 열린 프로젝트가 없습니다.")
                return None
            # 첫 프로젝트 활성화
            proj_name = projs[0]
            active_proj = desktop.open_project(proj_name)
        else:
            proj_name = active_proj.GetName()
    except Exception as e:
        print(f"❌ 프로젝트 정보 획득 실패: {e}")
        return None

    # 활성 디자인 확인 → Maxwell 2D인지 확인
    try:
        active_design = desktop.active_design()
        design_name = None
        design_type = None
        if active_design:
            design_name = active_design.GetName()
            design_type = active_design.GetDesignType()

        if not active_design or (design_type and "Maxwell" not in design_type) or (design_type and "2D" not in design_type):
            # 프로젝트의 디자인 목록에서 Maxwell 2D 탐색
            try:
                design_list = active_proj.GetTopDesignList()
            except Exception:
                design_list = []
            maxwell2d_name = None
            for dn in design_list:
                try:
                    d = active_proj.SetActiveDesign(dn)
                    # SetActiveDesign 반환이 None일 수 있으므로 다시 active_design 가져오기
                    ad = desktop.active_design()
                    if ad and "Maxwell" in ad.GetDesignType() and "2D" in ad.GetDesignType():
                        maxwell2d_name = ad.GetName()
                        break
                except Exception:
                    continue
            if not maxwell2d_name:
                print("❌ Maxwell 2D 디자인을 찾지 못했습니다.")
                return None
            design_name = maxwell2d_name
    except Exception as e:
        print(f"❌ 디자인 정보 획득 실패: {e}")
        return None

    # Maxwell2d 객체 attach
    try:
        m2d_attached = Maxwell2d(
            project=proj_name,
            design=design_name,
            version=aedt_version,
            new_desktop=False,
            non_graphical=non_graphical
        )
        print(f"✅ 연결 성공: Project='{proj_name}', Design='{design_name}'")
        return m2d_attached
    except Exception as e:
        print(f"❌ Maxwell2d attach 실패: {e}")
        return None

In [14]:
# 사용 예시
print("="*70)
print("🔌 현재 실행 중 Maxwell2d 객체 가져오기")
print("="*70)

m2d_running = get_running_maxwell2d()
if m2d_running:
    print(f"Design Type: {m2d_running.design_type}")
    print(f"Variables: {list(m2d_running.variable_manager.variables.keys())[:5]} ...")
else:
    print("⚠️ Maxwell2d 객체를 가져오지 못했습니다.")

🔌 현재 실행 중 Maxwell2d 객체 가져오기
PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT WARNING: Argument `specified_version` is deprecated for method `__init__`; use `version` instead.
PyAEDT WARNING: Argument `new_desktop_session` is deprecated for method `__init__`; use `new_desktop` instead.
PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Project e10_tutorial_ANSYSEM_2D set to active.
PyAEDT INFO: Aedt Objects correctly read
✅ 연결 성공: Project='e10_tutorial_ANSYSEM_2D', Design='Motor-CAD e10_tutorial_BPM_LabModel_1'
Design Type: Maxwell 2D
Variables: ['Is_2D_Design', 'DiaGap', 'DiaStatorYoke', 'DiaShaft', 'DiaShaft_Effective'] ...


In [15]:
m2d=m2d_running

### Get Design Information

In [16]:
designName=m2d.design_list
display(designName)


['Motor-CAD e10_tutorial',
 'Motor-CAD e10_tutorial_BPM_LabModel_1',
 'Motor-CAD e10_tutorial_BPM_LabModel_2']

In [18]:
m2d.set_active_design(designName[1])

PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Project e10_tutorial_ANSYSEM_2D set to active.
PyAEDT INFO: Aedt Objects correctly read


True

### Load Parametric Table

'F:/KDH/Thesis/JEET/e10_tuto/'

In [22]:
# ParametricTable Export 및 로드
import pandas as pd

sensitivity_setups = m2d.parametrics
parametricTablePath =os.path.join(m2d.project_path, "ParametricSetup1_Table.csv")

# AEDT에서 Parametric Table 추출
sensitivity_setups.setups[0].omodule.ExportParametricSetupTable("ParametricSetup1", parametricTablePath)

# CSV 파일 로드
ParametricTable = pd.read_csv(parametricTablePath)

print(f"✅ ParametricTable 로드 완료: {len(ParametricTable)}개의 variation")
display(ParametricTable.head())

✅ ParametricTable 로드 완료: 50개의 variation


,*,CurrentSweep,AngleSweep,SpeedSweep,WindingTempSweep,MagnetTempSweep
0,1,0A,0deg,500rpm,80cel,80cel
1,2,0.0A,12.857142857142858deg,500.0rpm,80.0cel,80.0cel
2,3,0.0A,25.714285714285715deg,500.0rpm,80.0cel,80.0cel
3,4,0.0A,38.57142857142857deg,500.0rpm,80.0cel,80.0cel
4,5,0.0A,51.42857142857143deg,500.0rpm,80.0cel,80.0cel


## FLD Export Function Definition

In [23]:
desktop = ansys.aedt.core.Desktop() 
pjtPath=desktop.project_path()
prjName=desktop.active_project().GetName()
filePath=os.path.join(pjtPath, prjName) 
pjt=desktop.load_project(filePath)



PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Project e10_tutorial_ANSYSEM_2D set to active.
PyAEDT INFO: Aedt Objects correctly read


In [32]:
model=m2d.modeler

In [33]:
intrinsics_dict = {"Time": "0.06s"}
"IntrinsicVar:=", f"Slice='2' {intrinsics_dict}",


('IntrinsicVar:=', "Slice='2' {'Time': '0.06s'}")

In [34]:
setup=pjt.get_setup(name='Setup1')

In [42]:
import ansys.aedt.core.visualization.plot.pyvista as pv_plot

In [45]:
pv_plot.ModelPlotter.fields

In [60]:
intrinsics_dict

{'Time': '0.06s'}

In [ ]:
post.create_fieldplot_surface(model['Rotor_1'].faces,quantity= "Mesh",plot_name="Mesh_plot4",intrinsics={'Slice': '1', 'Time': '0.006s'})    

In [57]:

post=m2d.post
plot = post.create_fieldplot_surface(model['Rotor_1'].faces,quantity= "Mesh",plot_name="Mesh_plot",intrinsics= f"Slice:='2' {intrinsics_dict}")


PyAEDT INFO: Plot Mesh_plot exists. returning the object.


In [ ]:
oModule.CreateFieldPlot(
	[
		"NAME:Mesh1",
		"SolutionName:="	, "Setup1 : Transient",
		"UserSpecifyName:="	, 0,
		"UserSpecifyFolder:="	, 0,
		"QuantityName:="	, "Mesh",
		"PlotFolder:="		, "MeshPlots",
		"FieldType:="		, "Fields",
		"StreamlinePlot:="	, False,
		"AdjacentSidePlot:="	, False,
		"FullModelPlot:="	, True,
		"IntrinsicVar:="	, "Slice=\'1\' Time=\'0.059999999999999998s\'",
		"PlotGeomInfo:="	, [1,"Surface","FacesList",37,"Stator_Lamination_Primitive","Ph1_P1_C4_1","Ph1_P1_C8_1","Ph1_P1_C12_1","Ph1_P1_C16_1","Ph1_P1_C20_1","Ph1_P1_C24_1","Ph2_P1_C3_1","Ph2_P1_C7_1","Ph2_P1_C11_1","Ph2_P1_C15_1","Ph2_P1_C19_1","Ph2_P1_C23_1","Ph3_P2_C4_1","Ph3_P2_C8_1","Ph3_P2_C12_1","Ph3_P2_C16_1","Ph3_P2_C20_1","Ph3_P2_C24_1","Ph1_P2_C1_2","Ph1_P2_C5_2","Ph1_P2_C9_2","Ph1_P2_C13_2","Ph1_P2_C17_2","Ph1_P2_C21_2","Ph2_P2_C2_2","Ph2_P2_C6_2","Ph2_P2_C10_2","Ph2_P2_C14_2","Ph2_P2_C18_2","Ph2_P2_C22_2","Ph3_P1_C2_2","Ph3_P1_C6_2","Ph3_P1_C10_2","Ph3_P1_C14_2","Ph3_P1_C18_2","Ph3_P1_C22_2"],
		"FilterBoxes:="		, [0],
		"Real time mode:="	, True,
		[
			"NAME:MeshSettings",
			"ShadingType:="		, 0,
			"Scale factor:="	, 100,
			"Transparency:="	, 0,
			"Mesh type:="		, "Shaded",
			"Surface only:="	, True,
			"Add grid:="		, True,
			"Refinement:="		, 0,
			"Use geometry color:="	, True,
			"Mesh line color:="	, [0,0,255],
			"Filled color:="	, [255,255,255]
		],
		"EnableGaussianSmoothing:=", False,
		"SurfaceOnly:="		, False
	], "Field")

In [55]:
m2d.project_path

'F:/KDH/Thesis/JEET/e10_tuto/'

In [54]:
 file_to_add = post.export_field_plot(plot.name, os.path.join(m2d.project_path, "Mesh_plot.aedtplt"))

PyAEDT ERROR: aedtplt file format is not supported for this plot.


In [ ]:
# m2d.post.export_mesh_obj(intrinsics={"Time":"0.06s"},export_air_objects=True,)
on_surfaces=True
export_air_objects=True
mesh_list = []
for el in part_names[1:3]:
    object3d = model[el]
    if on_surfaces:
        if not object3d.is3d or (not export_air_objects and object3d.material_name not in ["vacuum", "air"]):
            mesh_list += [i.id for i in object3d.faces]
    else:
        if not object3d.is3d or (not export_air_objects and object3d.material_name not in ["vacuum", "air"]):
            mesh_list.append(el)
    if on_surfaces:
        plot = post.create_fieldplot_surface(mesh_list, "Mesh", m2d.design_setups, {"Time":"0.06s"})
    else:
        plot = post.create_fieldplot_volume(mesh_list, "Mesh", m2d.design_setups, {"Time":"0.06s"})

    if plot:
        file_to_add = post.export_field_plot(plot.name, project_path)


In [ ]:
file_to_add

In [ ]:
mesh_plot = m2d.post.export_mesh_ob

In [ ]:
def export_fld_for_variations(
    m2d_obj,
    
    parametric_table,
    quantity={"Mesh","Mag_B"},
    solution="Setup1 : Transient",
    assignment="AllObjects",
    objects_type="Surf",
    output_dir=None,
    intrinsics=None,
    sweep_params=None
):
    """
    ParametricTable의 각 variation에 대해 FLD 파일을 export합니다.
    post_common_3d.py의 export_field_file 로직을 정확히 따릅니다.
    
    Parameters
    ----------
    m2d_obj : Maxwell2d
        Maxwell 2D 객체
    parametric_table : pandas.DataFrame
        Parametric sweep 결과 테이블
    quantity : str
        Export할 필드 quantity (예: "Mag_B", "Mag_H")
    solution : str
        Solution 이름 (예: "Setup1 : Transient")
    assignment : str or list
        Export할 객체 이름 ("AllObjects" 또는 객체 리스트)
    objects_type : str
        객체 타입 ("Surf", "Vol", "Line")
    output_dir : str or Path
        출력 디렉토리 경로 (None이면 working_directory 사용)
    intrinsics : dict
        Intrinsic 변수 (예: {"Time": "0.06s", "Phase": "0deg"})
    sweep_params : list
        Sweep 파라미터 이름 리스트 (None이면 자동 감지)
    
    Returns
    -------
    list
        생성된 FLD 파일 경로 리스트
    
    Examples
    --------
    >>> fld_files = export_fld_for_variations(
    ...     m2d_obj=m2d,
    ...     parametric_table=ParametricTable,
    ...     quantity="Mag_B",
    ...     solution="Setup1 : Transient",
    ...     intrinsics={"Time": "0.06s", "Phase": "0deg"}
    ... )
    """
    from pathlib import Path
    import re
    
    # 출력 디렉토리 설정
    if output_dir is None:
        output_dir = Path(m2d_obj.working_directory) / "FLD_Exports"
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Sweep 파라미터 자동 감지
    if sweep_params is None:
        sweep_params = ['AngleSweep', 'CurrentSweep', 'MagnetTempSweep', 
                        'SpeedSweep', 'WindingTempSweep']
    available_params = [col for col in sweep_params if col in parametric_table.columns]
    
    # Intrinsics 기본값 설정 (2D Transient의 경우 Time만 사용)
    if intrinsics is None:
        intrinsics_dict = {"Time": "0.06s"}
    elif isinstance(intrinsics, dict):
        intrinsics_dict = intrinsics.copy()
    else:
        intrinsics_dict = {}
    
    # FieldsReporter 모듈 가져오기
    ofieldsreporter = m2d_obj.odesign.GetModule("FieldsReporter")
    
    # 결과 저장 리스트
    fld_files = []
    
    print("=" * 70)
    print(f"📊 FLD Export: {quantity}")
    print(f"📁 출력 디렉토리: {output_dir}")
    print(f"📋 총 {len(parametric_table)}개의 Variation")
    print("=" * 70)
    
    # 각 variation에 대해 반복
    for idx, row in parametric_table.iterrows():
        print(f"\n{'='*70}")
        print(f"🔹 Variation {idx + 1}/{len(parametric_table)}")
        print(f"{'='*70}")
        
        # 1. Design 변수 값 변경 (AEDT 디자인에 직접 적용)
        param_info = {}
        
        for param in available_params:
            if param in row.index:
                value = row[param]
                param_info[param] = value
                
                # Design 변수 업데이트
                try:
                    m2d_obj[param] = str(value)
                    print(f"  {param}: {value} ✓")
                except Exception as e:
                    print(f"  ❌ {param} 변경 실패: {e}")
        
        print(f"\n  📋 Intrinsics: {intrinsics_dict}")
        
        # 3. FLD 파일명 생성
        fld_filename = f"{quantity}_variation_{idx + 1}"
        for param, value in param_info.items():
            # "Sweep" 문자 제거 및 숫자만 추출
            param_clean = param.replace('Sweep', '')
            numbers = re.findall(r'-?\d+\.?\d*', str(value))
            if numbers:
                clean_value = str(int(float(numbers[0])))
            else:
                clean_value = "0"
            fld_filename += f"_{param_clean}_{clean_value}"
        fld_filename += ".fld"
        fld_filepath = output_dir / fld_filename
        
        # 4. m2d.post.export_field_file 사용 (고수준 API)
        try:
            result = m2d_obj.post.export_field_file(
                quantity=quantity,
                solution=solution,
                variations=None,  # Design 변수를 이미 직접 변경했으므로 None
                output_file=str(fld_filepath),
                assignment=assignment,
                objects_type=objects_type,
                intrinsics=intrinsics_dict  # Time, Phase 등
            )
            
            # 파일 생성 확인
            if fld_filepath.exists():
                size_kb = fld_filepath.stat().st_size / 1024
                print(f"  ✅ FLD 파일 생성 완료: {fld_filepath.name} ({size_kb:.2f} KB)")
                fld_files.append(str(fld_filepath))
            else:
                print(f"  ❌ FLD 파일 생성 실패 (result={result})")
            
        except Exception as e:
            print(f"  ❌ FLD Export 실패: {e}")
    
    print(f"\n{'='*70}")
    print("✅ 모든 FLD Export 작업 완료")
    print(f"📁 저장 위치: {output_dir}")
    print(f"📦 총 {len(fld_files)}개의 FLD 파일 생성")
    print(f"{'='*70}")
    
    return fld_files


print("\n💡 주요 기능:")
print("✅ export_fld_for_variations 함수 정의 완료!")
print("  - ParametricTable의 각 variation에 대해 FLD 파일 생성")
print("  - Design 변수를 자동으로 변경하며 export")
print("  - Time/Phase 등 intrinsics 지원")
print("\n📝 사용법은 아래 예제 셀을 참고하세요.")

### Example: Export FLD for Single Variation (Test)

### Example: Export FLD for All Variations

In [ ]:
# 전체 ParametricTable에 대해 FLD export
# 주의: 시간이 오래 걸릴 수 있습니다!

print("=" * 70)
print(f"🚀 전체 {len(ParametricTable)}개 Variation FLD Export")
print("=" * 70)

# FLD export 실행
fld_files_all = export_fld_for_variations(
    m2d_obj=m2d,
    parametric_table=ParametricTable[1:3],  # 전체 테이블
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    objects_type="Surf",
    output_dir=r"D:\KDHe10\e10_example\FLD_Exports_All",
    intrinsics={"Time": "0.06s"}
)

print(f"\n{'='*70}")
print(f"✅ 완료! 총 {len(fld_files_all)}개의 FLD 파일 생성")
print(f"{'='*70}")

## Performance Analysis

### 성능 차이 분석: 단일 export vs export_fld_for_variations

In [ ]:
"""
🔍 왜 export_fld_for_variations가 훨씬 오래 걸릴까?

단일 export_field_file:
========================
- 1회 FLD export
- 약 3-5초 소요 (e10 모델 기준)

export_fld_for_variations:
==========================
- N개의 variation에 대해 반복
- 각 variation마다:
  1. Design 변수 변경: m2d[param] = value (약 0.1-0.5초)
  2. FLD export: export_field_file() (약 3-5초)
  3. 파일 검증 및 로깅 (약 0.1초)

총 소요 시간 = N × (변수변경시간 + FLD생성시간 + 검증시간)
            = N × (0.1~0.5s + 3~5s + 0.1s)
            = N × 3.2~5.6초

예: ParametricTable에 100개 variation이 있다면
    → 100 × 4초 = 400초 (약 6.7분)

💡 성능 최적화 방법:
==================
1. 병렬 처리 (여러 variation을 동시에 처리)
2. 필요한 variation만 선택적으로 export
3. 더 낮은 해상도/메쉬로 빠른 export
4. AEDT COM API 오버헤드 최소화
"""

print("=" * 70)
print("📊 성능 분석 정보")
print("=" * 70)
print(f"\n현재 ParametricTable: {len(ParametricTable)}개의 variation")
print(f"\n예상 소요 시간:")
print(f"  - 단일 FLD export: 약 3-5초")
print(f"  - 전체 variation export: 약 {len(ParametricTable) * 4} 초 ({len(ParametricTable) * 4 / 60:.1f}분)")
print(f"\n💡 시간이 오래 걸리는 이유:")
print(f"  1. 각 variation마다 Design 변수 변경")
print(f"  2. 각 변경 후 AEDT 내부 재계산")
print(f"  3. Field data를 디스크에 쓰는 I/O 시간")
print(f"  4. N번 반복 (N = {len(ParametricTable)})")
print("=" * 70)

### 실제 성능 측정: 단일 Export

In [ ]:
# 실행 시간 측정: 단일 FLD export
import time
import os

start = time.perf_counter()
fld_path = r"D:\KDHe10\e10_example\FLD_Exports_Test\Mag_B_single_test.fld"

try:
    result = m2d.post.export_field_file(
        quantity="Mag_B",
        assignment="AllObjects",
        output_file=fld_path,
        intrinsics={"Time": "0.06s"}
    )
    elapsed = time.perf_counter() - start
    print(f"✅ 단일 export_field_file 완료")
    print(f"⏱️  소요 시간: {elapsed:.3f} 초")
    print(f"📋 함수 반환값: {result}")
    
    if os.path.exists(fld_path):
        size_kb = os.path.getsize(fld_path) / 1024
        print(f"📦 파일 크기: {size_kb:.2f} KB")
        print(f"💾 파일 경로: {fld_path}")
    else:
        print("⚠️ 파일이 생성되지 않았습니다.")
        
except Exception as e:
    elapsed = time.perf_counter() - start
    print(f"❌ 실행 중 오류 발생 - 경과 시간: {elapsed:.3f} 초")
    print(f"오류 메시지: {e}")

### 실제 성능 측정: 3개 Variation Export

In [ ]:
# 실행 시간 측정: 3개 variation export (비교 테스트)
start_total = time.perf_counter()

# 처음 3개 variation만 추출
test_table_3 = ParametricTable.iloc[:3]

print("=" * 70)
print(f"⏱️  3개 Variation FLD Export 시간 측정")
print("=" * 70)

# FLD export 실행
fld_files_test_3 = export_fld_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_3,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    objects_type="Surf",
    output_dir=r"D:\KDHe10\e10_example\FLD_Exports_Performance_Test",
    intrinsics={"Time": "0.06s"}
)

elapsed_total = time.perf_counter() - start_total

print(f"\n{'='*70}")
print("📊 성능 측정 결과")
print(f"{'='*70}")
print(f"✅ 총 소요 시간: {elapsed_total:.3f} 초 ({elapsed_total/60:.2f} 분)")
print(f"📦 생성된 파일: {len(fld_files_test_3)}개")
print(f"⏱️  파일당 평균 시간: {elapsed_total / len(fld_files_test_3):.3f} 초")
print(f"\n💡 전체 {len(ParametricTable)}개 variation 예상 시간:")
print(f"   약 {(elapsed_total / 3) * len(ParametricTable):.1f} 초 ({(elapsed_total / 3) * len(ParametricTable) / 60:.1f} 분)")
print(f"{'='*70}")

## 성능 비교: FLD vs AEDTPLT

**왜 AEDTPLT가 더 빠른가?**

| 형식 | 파일 타입 | Export 시간 | 특징 |
|------|-----------|-------------|------|
| `.fld` | ASCII 텍스트 | 느림 (3-5초) | - 사람이 읽을 수 있음<br>- 파일 크기 큼<br>- Field Calculator 사용 |
| `.aedtplt` | Binary | 빠름 (1-2초) | - 바이너리 형식<br>- 파일 크기 작음<br>- AEDT 네이티브 형식 |

**속도 차이:**
- FLD: ~4초/variation → 100개 = 400초 (6.7분)
- AEDTPLT: ~1.5초/variation → 100개 = 150초 (2.5분) ✨

**약 2.7배 빠름!**

In [ ]:
designName

In [ ]:
m2d.set_active_design(designName[1])


In [ ]:
# ⏱️ 시간 측정 시작
import time
start_time = time.perf_counter()

print("=" * 70)
print("🔧 Field Plot 생성 및 AEDTPLT Export 테스트")
print("=" * 70)

oDesign = m2d.odesign
oDesign.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:LocalVariableTab",
			[
				"NAME:PropServers", 
				"LocalVariables"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:CurrentSweep",
					"Value:="		, "0A"
				],
				[
					"NAME:AngleSweep",
					"Value:="		, "77.1428571428571deg"
				],
				[
					"NAME:SpeedSweep",
					"Value:="		, "500rpm"
				],
				[
					"NAME:WindingTempSweep",
					"Value:="		, "80cel"
				],
				[
					"NAME:MagnetTempSweep",
					"Value:="		, "80cel"
				]
			]
		]
	])

elapsed_change_prop = time.perf_counter() - start_time
print(f"\n✅ Step 1: ChangeProperty 완료 ({elapsed_change_prop:.3f}초)")

oModule = oDesign.GetModule("FieldsReporter")
start_create_plot = time.perf_counter()

oModule.CreateFieldPlot(
	[
		"NAME:Mag_B1",
		"SolutionName:="	, "Setup1 : Transient",
		"UserSpecifyName:="	, 0,
		"UserSpecifyFolder:="	, 0,
		"QuantityName:="	, "Mag_B",
		"PlotFolder:="		, "B",
		"StreamlinePlot:="	, False,
		"AdjacentSidePlot:="	, False,
		"FullModelPlot:="	, False,
		"IntrinsicVar:="	, "Slice=\'2\' Time=\'0.059999999999999998s\'",
		"PlotGeomInfo:="	, [2,"Surface","FacesList",49,"Stator_Lamination_Primitive","Rotor_1","Rotor_Pocket_10","L2_1Magnet1N1_1_1","Rotor_Pocket_16","L2_1Magnet2N1_1_1","Rotor_Pocket_24","L1_1Magnet1N1_1_1","Rotor_Pocket_30","L1_1Magnet2N1_1_1","Shaft","Ph1_P1_C4_1","Ph1_P1_C8_1","Ph1_P1_C12_1","Ph1_P1_C16_1","Ph1_P1_C20_1","Ph1_P1_C24_1","Ph2_P1_C3_1","Ph2_P1_C7_1","Ph2_P1_C11_1","Ph2_P1_C15_1","Ph2_P1_C19_1","Ph2_P1_C23_1","Ph3_P2_C4_1","Ph3_P2_C8_1","Ph3_P2_C12_1","Ph3_P2_C16_1","Ph3_P2_C20_1","Ph3_P2_C24_1","Ph1_P2_C1_2","Ph1_P2_C5_2","Ph1_P2_C9_2","Ph1_P2_C13_2","Ph1_P2_C17_2","Ph1_P2_C21_2","Ph2_P2_C2_2","Ph2_P2_C6_2","Ph2_P2_C10_2","Ph2_P2_C14_2","Ph2_P2_C18_2","Ph2_P2_C22_2","Ph3_P1_C2_2","Ph3_P1_C6_2","Ph3_P1_C10_2","Ph3_P1_C14_2","Ph3_P1_C18_2","Ph3_P1_C22_2","Rotating_Band","Whole_Region","Line",2,"Boundary_Independent","Boundary_Dependent"],
		"FilterBoxes:="		, [0],
		[
			"NAME:PlotOnLineSettings",
			[
				"NAME:LineSettingsID",
				"Width:="		, 4,
				"Style:="		, "Cylinder"
			],
			"ShadingType:="		, 0,
			"IsoValType:="		, "Tone",
			"ArrowUniform:="	, False,
			"NumofArrow:="		, 100,
			"Refinement:="		, 0
		],
		[
			"NAME:PlotOnSurfaceSettings",
			"ShadingType:="		, 0,
			"Filled:="		, False,
			"IsoValType:="		, "Tone",
			"AddGrid:="		, False,
			"MapTransparency:="	, True,
			"Refinement:="		, 0,
			"Transparency:="	, 0,
			"SmoothingLevel:="	, 0,
			[
				"NAME:Arrow3DSpacingSettings",
				"ArrowUniform:="	, True,
				"ArrowSpacing:="	, 0,
				"MinArrowSpacing:="	, 0,
				"MaxArrowSpacing:="	, 0
			],
			"GridColor:="		, [255,255,255]
		],
		"EnableGaussianSmoothing:=", False,
		"SurfaceOnly:="		, False
	], "Field")

elapsed_create_plot = time.perf_counter() - start_create_plot
print(f"✅ Step 2: CreateFieldPlot 완료 ({elapsed_create_plot:.3f}초)")


In [ ]:
m2d.post.export_field_plot(plot_name="Mag_B1", output_dir="D:\\KDHe10\\e10_example\\FLD_Exports", file_name="Mag_B_Plot1_exported.aedtplt")

In [ ]:

start_export = time.perf_counter()
oModule.ExportFieldPlot("Mag_B1", False, "D:\\KDHe10\\e10_example\\FLD_Exports\\test.aedtplt")
elapsed_export = time.perf_counter() - start_export

# 총 시간
total_time = time.perf_counter() - start_time

# 파일 생성 확인
from pathlib import Path
output_file = Path("D:\\KDHe10\\e10_example\\FLD_Exports\\test.aedtplt")
if output_file.exists():
    size_kb = output_file.stat().st_size / 1024
    print(f"✅ Step 3: ExportFieldPlot 완료 ({elapsed_export:.3f}초)")
    print(f"   파일 크기: {size_kb:.2f} KB")
else:
    print(f"❌ Step 3: ExportFieldPlot 실패 ({elapsed_export:.3f}초)")

print(f"\n{'='*70}")
print("📊 실행 시간 분석")
print(f"{'='*70}")
print(f"  1️⃣ ChangeProperty:    {elapsed_change_prop:.3f}초 ({elapsed_change_prop/total_time*100:.1f}%)")
print(f"  2️⃣ CreateFieldPlot:   {elapsed_create_plot:.3f}초 ({elapsed_create_plot/total_time*100:.1f}%)")
print(f"  3️⃣ ExportFieldPlot:   {elapsed_export:.3f}초 ({elapsed_export/total_time*100:.1f}%)")
print(f"  {'─'*68}")
print(f"  ⏱️  총 소요 시간:      {total_time:.3f}초")
print(f"{'='*70}")

In [ ]:
# 결과 요약
print("\n📊 측정 결과 요약:")
print(f"  - ChangeProperty:   {elapsed_change_prop:.3f}초")
print(f"  - CreateFieldPlot:  {elapsed_create_plot:.3f}초") 
print(f"  - ExportFieldPlot:  {elapsed_export:.3f}초")
print(f"  - 총 소요 시간:     {total_time:.3f}초")

In [ ]:
intrinsics_dict

In [ ]:
intrinsic_str = " ".join([f"{k}='{v}'" for k, v in intrinsics_dict.items()])


## expt aedtplt

In [ ]:
def export_aedtplt_for_variations(
    m2d_obj,
    parametric_table,
    quantity="Mag_B",
    solution="Setup1 : Trintrinsics_dictnsient",
    assignment="AllObjects",
    output_dir=None,
    intrinsics=None,
    sweep_params=None
):
    """
    ParametricTable의 각 variation에 대해 AEDTPLT 파일을 export합니다.
    
    ⚡ 최적화 전략:
    - Field Plot을 최초 1회만 생성 (oModule.CreateFieldPlot)
    - 각 variation마다 변수만 변경 후 export
    - PyAEDT export_field_plot 사용 (빠름!)
    
    Parameters
    ----------
    m2d_obj : Maxwell2d
        Maxwell 2D 객체
    parametric_table : pandas.DataFrame
        Parametric sweep 결과 테이블
    quantity : str
        Export할 필드 quantity (예: "Mag_B", "Mag_H")
    solution : str
        Solution 이름 (예: "Setup1 : Transient")
    assignment : str
        Export할 객체 이름 ("AllObjects" 또는 객체 이름)
    output_dir : str or Path
        출력 디렉토리 경로 (None이면 working_directory 사용)
    intrinsics : dict
        Intrinsic 변수 (예: {"Time": "0.06s"})
    sweep_params : list
        Sweep 파라미터 이름 리스트 (None이면 자동 감지)
    
    Returns
    -------
    list
        생성된 AEDTPLT 파일 경로 리스트
    """
    from pathlib import Path
    import re
    import time as time_module
    
    # 출력 디렉토리 설정
    if output_dir is None:
        output_dir = Path(m2d_obj.working_directory) / "AEDTPLT_Exports"
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Sweep 파라미터 자동 감지
    if sweep_params is None:
        sweep_params = ['AngleSweep', 'CurrentSweep', 'MagnetTempSweep', 
                        'SpeedSweep', 'WindingTempSweep']
    available_params = [col for col in sweep_params if col in parametric_table.columns]
    
    # Intrinsics 기본값 설정
    if intrinsics is None:
        intrinsics_dict = {"Time": "0.06s"}
    elif isinstance(intrinsics, dict):
        intrinsics_dict = intrinsics.copy()
    else:
        intrinsics_dict = {}
    
    # COM API 객체 (함수 내부 로컬 변수)
    _oDesign = m2d_obj.odesign
    _oModule = _oDesign.GetModule("FieldsReporter")
    
    # 결과 저장 리스트
    aedtplt_files = []
    
    
    # Field Plot 이름
    plot_name = f"{quantity}_FieldPlot_Export"
    
    # ========================================
    # Step 0: Field Plot을 최초 1회만 생성
    # ========================================
     
    # 기존 Plot 삭제
    try:
        _oModule.DeleteFieldPlot([plot_name])
        print(f"  🧹 기존 Field Plot '{plot_name}' 삭제")
    except:
        pass
    
    # IntrinsicVar 문자열 생성
    intrinsic_str = " ".join([f"{k}='{v}'" for k, v in intrinsics_dict.items()])
    
    start_create = time_module.perf_counter()
    
    try:
        # CreateFieldPlot (최초 1회만)
        _oModule.CreateFieldPlot(
            [
                "NAME:" + plot_name,
                "SolutionName:=", solution,
                "UserSpecifyName:=", 0,
                "UserSpecifyFolder:=", 0,
                "QuantityName:=", quantity,
                "PlotFolder:=", "B",
                "StreamlinePlot:=", False,
                "AdjacentSidePlot:=", False,
                "FullModelPlot:=", False,
                "IntrinsicVar:=", f"Slice='2' {intrinsic_str}",
                "PlotGeomInfo:=", [2, "Surface", "FacesList", 49, 
                                   "Stator_Lamination_Primitive", "Rotor_1", "Rotor_Pocket_10",
                                   "L2_1Magnet1N1_1_1", "Rotor_Pocket_16", "L2_1Magnet2N1_1_1",
                                   "Rotor_Pocket_24", "L1_1Magnet1N1_1_1", "Rotor_Pocket_30",
                                   "L1_1Magnet2N1_1_1", "Shaft", "Ph1_P1_C4_1", "Ph1_P1_C8_1",
                                   "Ph1_P1_C12_1", "Ph1_P1_C16_1", "Ph1_P1_C20_1", "Ph1_P1_C24_1",
                                   "Ph2_P1_C3_1", "Ph2_P1_C7_1", "Ph2_P1_C11_1", "Ph2_P1_C15_1",
                                   "Ph2_P1_C19_1", "Ph2_P1_C23_1", "Ph3_P2_C4_1", "Ph3_P2_C8_1",
                                   "Ph3_P2_C12_1", "Ph3_P2_C16_1", "Ph3_P2_C20_1", "Ph3_P2_C24_1",
                                   "Ph1_P2_C1_2", "Ph1_P2_C5_2", "Ph1_P2_C9_2", "Ph1_P2_C13_2",
                                   "Ph1_P2_C17_2", "Ph1_P2_C21_2", "Ph2_P2_C2_2", "Ph2_P2_C6_2",
                                   "Ph2_P2_C10_2", "Ph2_P2_C14_2", "Ph2_P2_C18_2", "Ph2_P2_C22_2",
                                   "Ph3_P1_C2_2", "Ph3_P1_C6_2", "Ph3_P1_C10_2", "Ph3_P1_C14_2",
                                   "Ph3_P1_C18_2", "Ph3_P1_C22_2", "Rotating_Band", "Whole_Region",
                                   "Line", 2, "Boundary_Independent", "Boundary_Dependent"],
                "FilterBoxes:=", [0],
                [
                    "NAME:PlotOnSurfaceSettings",
                    "ShadingType:=", 0,
                    "Filled:=", False,
                    "IsoValType:=", "Tone",
                    "AddGrid:=", False,
                    "MapTransparency:=", True,
                    "Refinement:=", 0,
                    "Transparency:=", 0,
                    "SmoothingLevel:=", 0,
                    "GridColor:=", [255, 255, 255]
                ],
                "EnableGaussianSmoothing:=", False,
                "SurfaceOnly:=", False
            ], "Field")
        
        elapsed_create = time_module.perf_counter() - start_create
        print(f"  ✅ Field Plot '{plot_name}' 생성 완료 ({elapsed_create:.3f}초)")
        
    except Exception as e:
        print(f"  ❌ Field Plot 생성 실패: {e}")
        print(f"  💡 Field Plot 없이 진행할 수 없습니다.")
        return []
    
    # ========================================
    # 각 variation에 대해 반복
    # ========================================
    total_change_time = 0
    total_export_time = 0
    
    for idx, row in parametric_table.iterrows():
        print(f"\n{'='*70}")
        print(f"🔹 Variation {idx + 1}/{len(parametric_table)}")
        print(f"{'='*70}")
        
        # Step 1: Design 변수 변경
        start_change = time_module.perf_counter()
        
        param_info = {}
        changed_props = []
        
        for param in available_params:
            if param in row.index:
                value = row[param]
                param_info[param] = value
                changed_props.append([
                    "NAME:" + param,
                    "Value:=", str(value)
                ])
                print(f"  {param}: {value} ✓")
        
        # ChangeProperty 실행
        try:
            _oDesign.ChangeProperty(
                [
                    "NAME:AllTabs",
                    [
                        "NAME:LocalVariableTab",
                        ["NAME:PropServers", "LocalVariables"],
                        ["NAME:ChangedProps", *changed_props]
                    ]
                ]
            )
            elapsed_change = time_module.perf_counter() - start_change
            total_change_time += elapsed_change
            print(f"  ⏱️  ChangeProperty: {elapsed_change:.3f}초")
        except Exception as e:
            print(f"  ❌ 변수 변경 실패: {e}")
            continue
        
        # Step 2: AEDTPLT Export (PyAEDT 고수준 API 사용)
        start_export = time_module.perf_counter()
        
        # 파일명 생성 (확장자 제외 - export_field_plot이 자동으로 추가함)
        aedtplt_filename = f"{quantity}_variation_{idx + 1}"
        for param, value in param_info.items():
            param_clean = param.replace('Sweep', '')
            numbers = re.findall(r'-?\d+\.?\d*', str(value))
            if numbers:
                clean_value = str(int(float(numbers[0])))
            else:
                clean_value = "0"
            aedtplt_filename += f"_{param_clean}_{clean_value}"
        # .aedtplt 확장자는 export_field_plot()이 자동으로 추가
        
        try:
            # PyAEDT의 export_field_plot 사용 (빠름!)
            result = m2d_obj.post.export_field_plot(
                plot_name=plot_name,
                output_dir=str(output_dir),
                file_name=aedtplt_filename
            )
            
            elapsed_export = time_module.perf_counter() - start_export
            total_export_time += elapsed_export
            
            # 파일 확인 (export_field_plot이 .aedtplt 확장자를 자동 추가함)
            aedtplt_filepath = output_dir / (aedtplt_filename + ".aedtplt")
            if aedtplt_filepath.exists():
                size_kb = aedtplt_filepath.stat().st_size / 1024
                print(f"  ✅ AEDTPLT 생성: {aedtplt_filename} ({size_kb:.2f} KB, {elapsed_export:.3f}초)")
                aedtplt_files.append(str(aedtplt_filepath))
            else:
                print(f"  ❌ 파일 생성 실패 (result={result})")
                
        except Exception as e:
            print(f"  ❌ Export 실패: {e}")
    
    # Field Plot 정리
    try:
        _oModule.DeleteFieldPlot([plot_name])
        print(f"\n🧹 Field Plot '{plot_name}' 삭제 완료")
    except:
        pass
    
    # 성능 요약
    avg_change_time = total_change_time / len(parametric_table) if parametric_table.shape[0] > 0 else 0
    avg_export_time = total_export_time / len(parametric_table) if parametric_table.shape[0] > 0 else 0
    

    
    return aedtplt_files




### Example: Export AEDTPLT for Single Variation (Test)

In [ ]:
def create_mesh_field_plot(
    m2d_obj,
    name="Mesh1",
    solution="Setup1 : Transient",
    slice_id=1,
    time_value="0.06s",
    faces=None,
    plot_folder="MeshPlots",
    full_model=True,
    surface_only=False,
    scale_factor=100,
    add_grid=True,
    mesh_type="Shaded",
    transparency=0,
    use_geometry_color=True,
    mesh_line_color=(0, 0, 255),
    filled_color=(255, 255, 255)
):
    """Create (or recreate) a Mesh field plot via the low-level CreateFieldPlot API.

    Parameters
    ----------
    m2d_obj : Maxwell2d
        Active Maxwell2d object.
    name : str
        Field plot name (will be deleted/recreated if exists).
    solution : str
        Solution name string exactly as shown in AEDT (e.g. "Setup1 : Transient").
    slice_id : int | str
        Slice intrinsic value.
    time_value : str
        Time intrinsic value with units, e.g. "0.06s".
    faces : list[str] | None
        List of face/object names. If None, a default list (stator+windings) is used.
    plot_folder : str
        Folder under Project > Field Overlays to store the plot.
    full_model : bool
        Whether to mark FullModelPlot True.
    surface_only : bool
        SurfaceOnly flag at end of definition.
    scale_factor : int
        Mesh scale factor.
    add_grid : bool
        Draw grid overlay.
    mesh_type : str
        Mesh type string ("Shaded", etc.).
    transparency : int
        Transparency (0-100).
    use_geometry_color : bool
        Whether to use geometry colors.
    mesh_line_color : tuple[int,int,int]
        RGB mesh line color.
    filled_color : tuple[int,int,int]
        RGB filled color.

    Returns
    -------
    bool
        True if creation succeeded, False otherwise.
    """
    try:
        oDesign = m2d_obj.odesign
        oModule = oDesign.GetModule("FieldsReporter")
    except Exception as e:
        print(f"❌ Cannot access FieldsReporter module: {e}")
        return False

    # Default faces list (from existing Mesh CreateFieldPlot snippet) if not provided
    if faces is None:
        faces = [
            "Stator_Lamination_Primitive",
            "Ph1_P1_C4_1","Ph1_P1_C8_1","Ph1_P1_C12_1","Ph1_P1_C16_1","Ph1_P1_C20_1","Ph1_P1_C24_1",
            "Ph2_P1_C3_1","Ph2_P1_C7_1","Ph2_P1_C11_1","Ph2_P1_C15_1","Ph2_P1_C19_1","Ph2_P1_C23_1",
            "Ph3_P2_C4_1","Ph3_P2_C8_1","Ph3_P2_C12_1","Ph3_P2_C16_1","Ph3_P2_C20_1","Ph3_P2_C24_1",
            "Ph1_P2_C1_2","Ph1_P2_C5_2","Ph1_P2_C9_2","Ph1_P2_C13_2","Ph1_P2_C17_2","Ph1_P2_C21_2",
            "Ph2_P2_C2_2","Ph2_P2_C6_2","Ph2_P2_C10_2","Ph2_P2_C14_2","Ph2_P2_C18_2","Ph2_P2_C22_2",
            "Ph3_P1_C2_2","Ph3_P1_C6_2","Ph3_P1_C10_2","Ph3_P1_C14_2","Ph3_P1_C18_2","Ph3_P1_C22_2",
        ]
    faces_count = len(faces)

    # Delete existing plot with same name
    try:
        oModule.DeleteFieldPlot([name])
        print(f"🧹 기존 Mesh Plot '{name}' 삭제 완료")
    except Exception:
        pass  # ignore if not existing

    intrinsic_var = f"Slice='{slice_id}' Time='{time_value}'"

    definition = [
        f"NAME:{name}",
        "SolutionName:=", solution,
        "UserSpecifyName:=", 0,
        "UserSpecifyFolder:=", 0,
        "QuantityName:=", "Mesh",
        "PlotFolder:=", plot_folder,
        "FieldType:=", "Fields",
        "StreamlinePlot:=", False,
        "AdjacentSidePlot:=", False,
        "FullModelPlot:=", full_model,
        "IntrinsicVar:=", intrinsic_var,
        "PlotGeomInfo:=", [1, "Surface", "FacesList", faces_count, *faces],
        "FilterBoxes:=", [0],
        "Real time mode:=", True,
        [
            "NAME:MeshSettings",
            "ShadingType:=", 0,
            "Scale factor:=", scale_factor,
            "Transparency:=", transparency,
            "Mesh type:=", mesh_type,
            "Surface only:=", surface_only,
            "Add grid:=", add_grid,
            "Refinement:=", 0,
            "Use geometry color:=", use_geometry_color,
            "Mesh line color:=", list(mesh_line_color),
            "Filled color:=", list(filled_color),
        ],
        "EnableGaussianSmoothing:=", False,
        "SurfaceOnly:=", False,
    ]

    try:
        oModule.CreateFieldPlot(definition, "Field")
        print(f"✅ Mesh Field Plot '{name}' 생성 완료 (faces={faces_count}, slice={slice_id}, time={time_value})")
        return True
    except Exception as e:
        print(f"❌ Mesh Field Plot 생성 실패: {e}")
        return False

# 사용 예시 (실행하려면 주석 해제)
# create_mesh_field_plot(m2d, name="Mesh1", time_value="0.06s", slice_id=1)


In [ ]:
test_table_1 = ParametricTable.iloc[:1]

aedtplt_files = export_aedtplt_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_1,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    output_dir=r"D:\KDHe10\e10_example\AEDTPLT_Exports_Optimized",
    intrinsics={"Time": "0.06s"}
)

### Export All 50 Variations

In [11]:
# 전체 50개 variation AEDTPLT export
import time

print("=" * 70)
print(f"🚀 전체 {len(ParametricTable)}개 Variation AEDTPLT Export")
print("=" * 70)

start_all = time.perf_counter()

aedtplt_files_all = export_aedtplt_for_variations(
    m2d_obj=m2d,
    parametric_table=ParametricTable,  # 전체 50개
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    output_dir=r"D:\KDHe10\e10_example\AEDTPLT_Exports_All_50",
    intrinsics={"Time": "0.06s"}
)

elapsed_all = time.perf_counter() - start_all

print(f"\n{'='*70}")
print("🎉 전체 Export 완료!")
print(f"{'='*70}")
print(f"⏱️  총 소요 시간: {elapsed_all:.1f}초 ({elapsed_all/60:.1f}분)")
print(f"📦 생성된 파일: {len(aedtplt_files_all)}개")
print(f"⚡ 평균 시간: {elapsed_all/len(ParametricTable):.1f}초/variation")
print(f"{'='*70}")

In [ ]:
py_vista_plot = m3d.post.plot_field(
    quantity="Mag_B", assignment='Rotor_Lamination_Primitive', plot_cad_objs=True, show=False
)


In [ ]:
import ansys.aedt.core.visualization.plot.pyvista as pyvista

In [ ]:
modelplot = pyvista.ModelPlotter()

## ModelPlotter를 위한 FLD Export

**AEDTPLT → FLD 변환은 불가능합니다.**
- `.aedtplt`: Binary 형식 (AEDT 전용)
- `.fld`: ASCII 텍스트 (외부 도구 호환)

**해결책:** `export_fld_for_variations()` 함수로 FLD 파일 생성

```python
# ModelPlotter 사용 예시:
from ansys.aedt.core.visualization.plot.pyvista import ModelPlotter

model_plotter = ModelPlotter()
model_plotter.add_field_from_file(fld_file_path)
model_plotter.plot()
```

In [ ]:
# 예시: 첫 번째 variation에 대해 FLD 파일 생성
import time

test_table_for_fld = ParametricTable.iloc[:2]

print("=" * 70)
print("📊 FLD Export for ModelPlotter")
print("=" * 70)

start_fld = time.perf_counter()

fld_files_for_plotter = export_fld_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_for_fld,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    objects_type="Surf",
    output_dir=r"D:\KDHe10\e10_example\FLD_For_ModelPlotter",
    intrinsics={"Time": "0.06s"}
)

elapsed_fld = time.perf_counter() - start_fld

print(f"\n{'='*70}")
print(f"✅ FLD Export 완료!")
print(f"⏱️  소요 시간: {elapsed_fld:.3f}초")
print(f"📦 생성된 파일: {len(fld_files_for_plotter)}개")
print(f"{'='*70}")

if fld_files_for_plotter:
    print(f"\n💡 ModelPlotter 사용법:")
    print(f"```python")
    print(f"from ansys.aedt.core.visualization.plot.pyvista import ModelPlotter")
    print(f"model_plotter = ModelPlotter()")
    print(f"model_plotter.add_field_from_file(r'{fld_files_for_plotter[0]}')")
    print(f"model_plotter.plot()")
    print(f"```")

In [68]:
# ModelPlotter로 FLD 파일 시각화
from ansys.aedt.core.visualization.plot.pyvista import ModelPlotter

if fld_files_for_plotter:
    print("=" * 70)
    print("🎨 ModelPlotter 시각화")
    print("=" * 70)
    
    model_plotter = ModelPlotter()
    
    # FLD 파일 로드
    print(f"📂 파일 로드: {fld_files_for_plotter[0]}")
    model_plotter.add_field_from_file(fld_files_for_plotter[0])
    
    # Plot 생성
    print("🖼️  Plot 생성 중...")
    model_plotter.plot()
    
    print("✅ 시각화 완료!")
else:
    print("❌ FLD 파일이 없습니다. 먼저 위 셀을 실행하세요.")

NameError: name 'fld_files_for_plotter' is not defined

## FLD 파일 구조 분석

FLD 파일에 메쉬 정보가 포함되어 있는지 확인해봅시다.

In [ ]:
# PyVista 객체로 mesh+field 저장하기 (.vtp)
import pyvista as pv
from ansys.aedt.core.visualization.plot.pyvista import ModelPlotter
from pathlib import Path

print("="*70)
print("💾 PyVista .vtp로 저장 (mesh + field)")
print("="*70)

# AEDT에서 PyVista 플롯 객체 받기
pv_plot = m2d.post.plot_field(
    quantity="Mag_B",
    intrinsics={"Time": "0.06s"},
    assignment="Rotor_1",
    plot_cad_objs=True,
    show=False
)


In [ ]:
import ansys.aedt.core 
from pathlib import Path
import pyvista as pv

folder_path = Path(r"D:\KDHe10\e10_example")
file_path = folder_path.joinpath("e10_tutorial_ANSYSEM_2D.aedt")    
exports_folder_path = folder_path.joinpath("exports_folder")
exports_folder_path.mkdir(parents=True,exist_ok=True)


In [ ]:

mesh_folder = exports_folder_path.joinpath("mesh_folder")
mesh_folder.mkdir(parents=True,exist_ok=True)

result_folder = exports_folder_path.joinpath("result_folder")
result_folder.mkdir(parents=True,exist_ok=True)


In [ ]:
for name in part_names[:1]:
    obj =  model.get_object_from_name(name)

    #################### Export Mesh Plot #######################

    if obj.solve_inside:       # check for solve_inside
        mesh_plot = post.create_fieldplot_volume([name],quantity="Mesh",
                                                 plot_name="Mesh_plot")
    else:
        mesh_plot = post.create_fieldplot_surface(obj.faces,quantity="Mesh",
                                                  plot_name="Mesh_plot",
                                                  intrinsics=prj.setups[0].default_intrinsics)
    
    print(f"Exporting Mesh and H_Vector for: {name}")

    mesh_file_name = mesh_folder.joinpath(name+'_mesh')
    export_status = post.export_field_plot(plot_name = "Mesh_plot",
                            output_dir = exports_folder_path,
                            file_name = mesh_file_name.__str__(),
                            file_format = 'case')

    if export_status:
            print(f"\tExported Mesh")
    else:
        print(f"\tNot Exported Mesh")
    
    # mesh_plot.delete()

    #################### Export H Vector #######################
    if obj.solve_inside:
        B_field_plot = post.create_fieldplot_volume([name],quantity=[]"Vector_B",M,
                                                    plot_name="B_vector_plot")
    else:
        B_field_plot = post.create_fieldplot_surface(obj.faces,quantity="Vector_B",
                                                     plot_name="B_vector_plot",
                                                     intrinsics=prj.setups[0].default_intrinsics)


    result_file_name = result_folder.joinpath(name+'_B_vector')
    export_status = post.export_field_plot(plot_name = "B_vector_plot",
                            output_dir = exports_folder_path,
                            file_name = result_file_name.__str__(),
                            file_format = 'case')
    if export_status:
        print(f"\tExported B_Vector")
    else:
        print(f"\tNot Exported B_Vector")

    # B_field_plot.delete()